In [ ]:
# pip install fastapi uvicorn ollama openai python-multipart python-dotenv

import os
import io
import base64
from typing import Optional
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
import ollama
from openai import OpenAI
import uvicorn

# 환경 변수 로드
load_dotenv()

app = FastAPI()

# CORS 설정 (가이드 준수: 모든 허용)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def getModelResponse(imageContent: bytes, userQuestion: str):
    """ 
    설정된 모델(GPT 또는 OLLAMA)에 따라 이미지 분석 결과를 반환하는 함수입니다.
    """
    try:
        useModel = os.getenv("USE_MODEL", "OLLAMA")
        
        if useModel == "GPT":
            # OpenAI GPT-4o Vision 활용 (텍스트 추출 및 질문 처리)
            client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            base64Image = base64.b64encode(imageContent).decode('utf-8')
            
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": userQuestion},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64Image}",
                                },
                            },
                        ],
                    }
                ],
            )
            return response.choices[0].message.content
            
        elif useModel == "OLLAMA":
            # 로컬 Ollama gemma4:e2b 모델 활용
            ollamaModel = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
            response = ollama.chat(
                model=ollamaModel,
                messages=[{
                    'role': 'user',
                    'content': userQuestion,
                    'images': [imageContent]
                }]
            )
            return response['message']['content']
            
        else:
            return "지원하지 않는 모델 설정입니다."
            
    except Exception as e:
        raise e

@app.post("/analyze")
async def analyzeImage(image: UploadFile = File(...), question: str = Form(...)):
    """ 
    사용자가 업로드한 이미지와 질문을 받아 분석 결과를 JSON으로 반환하는 API 엔드포인트입니다.
    """
    try:
        # 이미지 데이터 읽기
        imageContent = await image.read()
        
        # 모델을 통한 분석 수행
        analysisResult = getModelResponse(imageContent, question)
        
        # 데이터 처리 예시 (가이드 준수: 리스트 컴프리헨션 금지 및 명시적 반복문 사용)
        # 여기서는 결과 텍스트를 단어 리스트로 변환하는 예시를 보여줍니다.
        wordList = analysisResult.split()
        processedWords = []
        for i in range(0, len(wordList)):
            processedWords.append(wordList[i])
            
        return {
            "success": True,
            "result": analysisResult,
            "metadata": {
                "fileName": image.filename,
                "wordCount": len(processedWords)
            }
        }
        
    except Exception as e:
        # 에러 발생 시 가이드에 정의된 JSON 형식 반환
        return {
            "success": false,
            "message": str(e)
        }

if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)